In [ ]:
# %% [markdown]
# Tier-2 Scraper (sequential + resume via SCRAPE_PROGRESS_INDEX, minimal logging)
# - Starts from SCRAPE_PROGRESS_INDEX in All_Tokens.env (0-based).
# - After each repo completes, updates SCRAPE_PROGRESS_INDEX to i+1.
# - Minimal console output: "[i/total] repo (token t/m) — start" and "done → SCRAPE_PROGRESS_INDEX=…".
# - Robust HTTP (HTTP/2 when available, otherwise HTTP/1.1; retries/backoff; rate-limit aware).
# - Writes runs/jobs/timing NDJSON per repo; optional episodes→runs Parquet mapping.

# %% Install dependencies (run once per environment)
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "httpx[http2]>=0.27.0", "pandas>=2.1.0", "pyarrow>=15.0.0", "nest_asyncio>=1.6.0"
])

# %% Configuration — set your paths and knobs here
from pathlib import Path

BASE = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ3")
URL_CSV = BASE / "URL_List.csv"
EPISODES_CSV = BASE / "episodes_enriched_combined.csv"          # optional mapping

# 👉 Tokens / progress file:
TOKENS_ENV = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

OUTDIR = BASE / "ci_out"                                         # output folder
SINCE = "2024-01-01"                                             # set None for full history

INCLUDE_JOBS = True
INCLUDE_TIMING = True
MAX_RUNS_PER_REPO = None                                         # None = no cap

MAP_EPISODES = True
WINDOW_DAYS = 14

# Minimal logging only
PRINT_START_SUMMARY = True   # one-liner summary at start
DEBUG_HTTP = False           # True → show retry/rate-limit logs; False → keep quiet

# Resume env variable name
PROGRESS_ENV_KEY = "SCRAPE_PROGRESS_INDEX"

print("Config OK")

# %% Utilities
import os, re, json, asyncio, random
from typing import List, Iterable, Dict, Any, Optional
from datetime import datetime, timezone
import pandas as pd

def ensure_dir(path: str | os.PathLike) -> None:
    os.makedirs(path, exist_ok=True)

def dump_ndjson(path: str | os.PathLike, rows: Iterable[Dict[str, Any]]) -> None:
    ensure_dir(os.path.dirname(str(path)))
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_env_tokens(env_path: str | os.PathLike, prefix: str = "GITHUB_TOKEN_") -> List[str]:
    tokens = []
    with open(env_path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            if k.upper().startswith(prefix) and v:
                tokens.append(v)
    if not tokens:
        raise RuntimeError(f"No tokens found in {env_path} with prefix {prefix}")
    return tokens

def parse_repo_from_text(s: str) -> Optional[str]:
    if not isinstance(s, str):
        return None
    s = s.strip()
    if re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$", s):
        return s
    m = re.search(r"github\.com/([A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+)", s, re.I)
    if m:
        return m.group(1)
    return None

def extract_repos_from_csv(url_csv: str | os.PathLike) -> List[str]:
    df = pd.read_csv(url_csv)
    repos = set()
    for col in df.columns:
        for v in df[col].astype(str):
            r = parse_repo_from_text(v)
            if r:
                repos.add(r)
    repos = sorted(repos)
    if not repos:
        raise RuntimeError("No GitHub repos parsed from URL CSV (need owner/name or GH URLs).")
    return repos

# (Used by optional mapping step)
def _robust_parse_epoch_series(s: pd.Series) -> pd.Series:
    snum = pd.to_numeric(s, errors="coerce")
    if snum.notna().sum() == 0:
        return pd.Series([], dtype="datetime64[ns, UTC]")
    med = snum.dropna().abs().median()
    unit = "us" if med > 1e14 else ("ms" if med > 1e11 else "s")
    return pd.to_datetime(snum, unit=unit, errors="coerce", utc=True)

print("Utils ready")

# %% Env helpers for progress (atomic updates)
def _read_env_var(env_path: str | os.PathLike, key: str) -> Optional[str]:
    key_re = re.compile(rf"^\s*{re.escape(key)}\s*=(.*)$", re.I)
    try:
        with open(env_path, "r", encoding="utf-8") as f:
            for raw in f:
                m = key_re.match(raw.strip())
                if m:
                    val = m.group(1).strip().strip('"').strip("'")
                    return val
    except FileNotFoundError:
        return None
    return None

def _write_env_var(env_path: str | os.PathLike, key: str, value: str) -> None:
    lines: List[str] = []
    found = False
    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as f:
            for raw in f:
                line = raw.rstrip("\n")
                if re.match(r"^\s*"+re.escape(key)+r"\s*=", line, re.I):
                    lines.append(f"{key}={value}")
                    found = True
                else:
                    lines.append(line)
    if not found:
        lines.append(f"{key}={value}")
    tmp = str(env_path) + ".tmp"
    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lines) + "\n")
    os.replace(tmp, env_path)

def get_progress_index(env_path: str | os.PathLike, key: str) -> int:
    v = _read_env_var(env_path, key)
    if v is None:
        return 0
    try:
        idx = int(str(v).strip())
        return max(0, idx)
    except Exception:
        return 0

def set_progress_index(env_path: str | os.PathLike, key: str, value: int) -> None:
    _write_env_var(env_path, key, str(int(value)))

def log_debug(msg: str) -> None:
    if DEBUG_HTTP:
        print(msg, flush=True)

# %% HTTP/2 availability check (fallback to HTTP/1.1 if h2 missing)
try:
    import h2  # noqa: F401
    _HTTP2_AVAILABLE = True
except Exception:
    _HTTP2_AVAILABLE = False
    if PRINT_START_SUMMARY:
        print("HTTP/2 not available (h2 not installed). Falling back to HTTP/1.1.")

# %% GitHub API client (async, robust retries)
import httpx

GITHUB_API = "https://api.github.com"

class GHClient:
    def __init__(self, token: str, rate_sleep: float = 0.2):
        self.token = token
        self.rate_sleep = rate_sleep
        self.headers = {
            "Authorization": f"Bearer {self.token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        }
        self.client = httpx.AsyncClient(
            headers=self.headers,
            http2=_HTTP2_AVAILABLE,  # fallback to HTTP/1.1 if not available
            timeout=httpx.Timeout(connect=30.0, read=60.0, write=30.0, pool=60.0),
            limits=httpx.Limits(max_connections=32, max_keepalive_connections=16, keepalive_expiry=30.0),
        )

    async def close(self):
        await self.client.aclose()

    async def _recreate_client(self):
        try:
            await self.client.aclose()
        except Exception:
            pass
        self.client = httpx.AsyncClient(
            headers=self.headers,
            http2=_HTTP2_AVAILABLE,
            timeout=httpx.Timeout(connect=30.0, read=60.0, write=30.0, pool=60.0),
            limits=httpx.Limits(max_connections=32, max_keepalive_connections=16, keepalive_expiry=30.0),
        )

    async def _get(self, url, params=None, max_retries: int = 5) -> httpx.Response:
        RETRYABLE = (
            httpx.RemoteProtocolError,
            httpx.ConnectError, httpx.ReadError, httpx.WriteError,
            httpx.PoolTimeout, httpx.TimeoutException,
        )
        last_exc: Optional[BaseException] = None
        for attempt in range(max_retries):
            try:
                resp = await self.client.get(url, params=params)
                # Upstream transient errors → retry
                if resp.status_code in (502, 503, 504, 429):
                    raise httpx.HTTPStatusError("upstream transient", request=resp.request, response=resp)

                # Rate limit handling (403 classic OR secondary rate limit)
                if resp.status_code == 403 and (
                    resp.headers.get("X-RateLimit-Remaining") == "0"
                    or "rate limit" in (resp.text or "").lower()
                ):
                    reset = resp.headers.get("X-RateLimit-Reset")
                    wait_s = 10
                    if reset:
                        try:
                            now = int(datetime.now(timezone.utc).timestamp())
                            wait_s = max(1, int(reset) - now + 1)
                        except Exception:
                            pass
                    log_debug(f"[rate-limit] sleeping {wait_s}s")
                    await asyncio.sleep(wait_s)
                    continue

                resp.raise_for_status()
                await asyncio.sleep(self.rate_sleep)
                return resp

            except (httpx.HTTPStatusError,) as e:
                last_exc = e
                backoff = min(20.0, (2 ** attempt)) + random.random()
                log_debug(f"[http-status] retry {attempt+1}/{max_retries} in {backoff:.1f}s: {url} ({e})")
                await asyncio.sleep(backoff)

            except RETRYABLE as e:
                last_exc = e
                backoff = min(20.0, (2 ** attempt)) + random.random()
                log_debug(f"[transport] {e.__class__.__name__} → retry {attempt+1}/{max_retries} in {backoff:.1f}s: {url}")
                await self._recreate_client()
                await asyncio.sleep(backoff)

        raise last_exc if last_exc else httpx.RemoteProtocolError("Unrecoverable transport error")

    # Repo-level runs (no 'created' filter; we filter locally)
    async def list_workflow_runs(self, owner: str, repo: str, per_page=100):
        page = 1
        while True:
            params = {"per_page": per_page, "page": page}
            resp = await self._get(f"{GITHUB_API}/repos/{owner}/{repo}/actions/runs", params=params)
            data = resp.json()
            runs = data.get("workflow_runs", [])
            if not runs:
                break
            for run in runs:
                yield run
            if len(runs) < per_page:
                break
            page += 1

    async def list_jobs_for_run(self, owner: str, repo: str, run_id: int):
        page = 1
        per_page = 100
        while True:
            resp = await self._get(
                f"{GITHUB_API}/repos/{owner}/{repo}/actions/runs/{run_id}/jobs",
                params={"per_page": per_page, "page": page}
            )
            data = resp.json()
            jobs = data.get("jobs", [])
            if not jobs:
                break
            for job in jobs:
                yield job
            if len(jobs) < per_page:
                break
            page += 1

    # Workflows & per-workflow runs (fallback)
    async def list_workflows(self, owner: str, repo: str):
        page = 1
        per_page = 100
        while True:
            resp = await self._get(
                f"{GITHUB_API}/repos/{owner}/{repo}/actions/workflows",
                params={"per_page": per_page, "page": page}
            )
            data = resp.json()
            wfs = data.get("workflows", [])
            if not wfs:
                break
            for wf in wfs:
                yield wf
            if len(wfs) < per_page:
                break
            page += 1

    async def list_workflow_runs_for_workflow(self, owner: str, repo: str, workflow_id: int, per_page=100):
        page = 1
        while True:
            resp = await self._get(
                f"{GITHUB_API}/repos/{owner}/{repo}/actions/workflows/{workflow_id}/runs",
                params={"per_page": per_page, "page": page}
            )
            data = resp.json()
            runs = data.get("workflow_runs", [])
            if not runs:
                break
            for run in runs:
                yield run
            if len(runs) < per_page:
                break
            page += 1

    async def get_run_timing(self, owner: str, repo: str, run_id: int):
        resp = await self._get(f"{GITHUB_API}/repos/{owner}/{repo}/actions/runs/{run_id}/timing")
        return resp.json()

print("GitHub client ready")

# %% Scrape logic (per-repo, since filter, fallback) — silent except errors
async def scrape_repo(owner_repo: str,
                      client: GHClient,
                      outdir: str | os.PathLike,
                      since_iso: Optional[str],
                      include_jobs: bool,
                      include_timing: bool,
                      max_runs: Optional[int]) -> None:
    owner, repo = owner_repo.split("/", 1)
    base = Path(outdir) / owner_repo.replace("/", "_")
    runs_path = base / "runs.ndjson"
    jobs_path = base / "jobs.ndjson"
    timing_path = base / "timing.ndjson"
    ensure_dir(base)

    since_dt = pd.to_datetime(since_iso, utc=True) if since_iso else None
    run_count = 0

    async def _consume_runs(run_iter):
        nonlocal run_count
        async for run in run_iter:
            # local since filter (runs are reverse-chronological)
            ca = run.get("run_started_at") or run.get("created_at")
            if since_dt is not None and ca:
                ca_dt = pd.to_datetime(ca, utc=True, errors="coerce")
                if pd.notna(ca_dt) and ca_dt < since_dt:
                    return "stopped_by_since"

            dump_ndjson(runs_path, [run])
            run_count += 1

            if include_jobs:
                try:
                    async for job in client.list_jobs_for_run(owner, repo, run["id"]):
                        dump_ndjson(jobs_path, [job])
                except httpx.HTTPError:
                    # continue on job fetch failures
                    pass

            if include_timing:
                try:
                    timing = await client.get_run_timing(owner, repo, run["id"])
                    dump_ndjson(timing_path, [dict(run_id=run["id"], **timing)])
                except httpx.HTTPError:
                    pass

            if max_runs and run_count >= max_runs:
                return "stopped_by_cap"

        return "exhausted"

    try:
        status = await _consume_runs(client.list_workflow_runs(owner, repo))
        if run_count == 0:
            async for wf in client.list_workflows(owner, repo):
                wf_id = wf.get("id")
                if not wf_id:
                    continue
                status = await _consume_runs(client.list_workflow_runs_for_workflow(owner, repo, wf_id))
                if max_runs and run_count >= max_runs:
                    break

    except Exception as e:
        ensure_dir(base)
        with open(base / "errors.log", "a", encoding="utf-8") as f:
            f.write(f"{datetime.now().isoformat()} error: {e}\n")
        raise

print("Scrape logic loaded")

# %% Sanity check tokens (masked) — optional one-liner
def _read_env_tokens(env_path, prefix="GITHUB_TOKEN_"):
    toks = []
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            if k.upper().startswith(prefix) and v:
                toks.append(v)
    return toks

try:
    _toks = _read_env_tokens(TOKENS_ENV)
    if PRINT_START_SUMMARY:
        print(f"Tokens: {len(_toks)} found; Progress key: {PROGRESS_ENV_KEY}")
except Exception as e:
    print(f"Token check failed: {e}")

# %% Execute scraping (sequential + strict resume from SCRAPE_PROGRESS_INDEX)
import nest_asyncio, asyncio
nest_asyncio.apply()
if sys.platform.startswith("win"):
    try:
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    except Exception:
        pass

since_iso = SINCE  # fixed start date; set to None to fetch all history

tokens = read_env_tokens(TOKENS_ENV)
repos = extract_repos_from_csv(URL_CSV)
TOTAL = len(repos)
ensure_dir(OUTDIR)
if PRINT_START_SUMMARY:
    print(f"Repos: {TOTAL}; Since: {since_iso or 'ALL'}; Output: {OUTDIR}")

async def run_all_sequential(
    repos: List[str],
    tokens: List[str],
    outdir: str | os.PathLike,
    since_iso: Optional[str],
    include_jobs: bool,
    include_timing: bool,
    max_runs: Optional[int],
    progress_env_path: str | os.PathLike,
    progress_key: str,
):
    clients = [GHClient(t) for t in tokens]
    start_idx = get_progress_index(progress_env_path, progress_key)
    print(f"Starting from {progress_key}={start_idx} ({min(start_idx+1, len(repos))}/{len(repos)})")

    if start_idx >= len(repos):
        print(f"{progress_key} ({start_idx}) >= repo count ({len(repos)}). Nothing to do.")
        await asyncio.gather(*(c.close() for c in clients))
        return

    try:
        for i in range(start_idx, len(repos)):
            repo = repos[i]
            tok_i = i % len(clients)  # rotation
            print(f"[{i+1}/{len(repos)}] {repo} (token {tok_i+1}/{len(clients)}) — start", flush=True)
            try:
                await scrape_repo(
                    owner_repo=repo,
                    client=clients[tok_i],
                    outdir=str(outdir),
                    since_iso=since_iso,
                    include_jobs=include_jobs,
                    include_timing=include_timing,
                    max_runs=max_runs
                )
            except Exception as e:
                # Do NOT advance index; leave it at i so re-run starts from this repo again
                with open(Path(outdir) / "top_level_errors.log", "a", encoding="utf-8") as f:
                    f.write(f"{datetime.now().isoformat()} repo={repo} idx={i} error={repr(e)}\n")
                print(f"[{i+1}/{len(repos)}] {repo} — ERROR; keeping {progress_key}={i}", flush=True)
                break

            # Repo finished successfully → advance index so it resumes at the next repo later
            set_progress_index(progress_env_path, progress_key, i + 1)
            print(f"[{i+1}/{len(repos)}] {repo} — done → {progress_key}={i+1}", flush=True)

        final_idx = get_progress_index(progress_env_path, progress_key)
        print(f"Run finished. Current {progress_key}={final_idx}")

    finally:
        await asyncio.gather(*(c.close() for c in clients))

await run_all_sequential(
    repos=repos,
    tokens=tokens,
    outdir=str(OUTDIR),
    since_iso=since_iso,
    include_jobs=INCLUDE_JOBS,
    include_timing=INCLUDE_TIMING,
    max_runs=MAX_RUNS_PER_REPO,
    progress_env_path=str(TOKENS_ENV),
    progress_key=PROGRESS_ENV_KEY,
)

# %% (Optional) Map CCE episodes → runs (Parquet)
if MAP_EPISODES and EPISODES_CSV.exists():
    def map_episodes_to_runs(episodes_csv: str | os.PathLike, outdir: str | os.PathLike, window_days: int = 14) -> pd.DataFrame:
        eps = pd.read_csv(episodes_csv)
        # detect repo column
        repo_col = None
        for k in ["repo", "repository", "full_name", "repo_full_name"]:
            m = [c for c in eps.columns if re.search(k, c, re.I)]
            if m:
                repo_col = m[0]; break
        if not repo_col:
            raise ValueError("Could not detect a repo column in episodes CSV")

        ts_col = None
        for k in ["timestamp", "time", "created", "episode_time", "event_time", "date", "datetime"]:
            m = [c for c in eps.columns if re.search(k, c, re.I)]
            if m:
                ts_col = m[0]; break
        if not ts_col:
            raise ValueError("Could not detect a timestamp column in episodes CSV")

        fam_col = None
        for k in ["family", "label", "cce_family"]:
            m = [c for c in eps.columns if re.search(k, c, re.I)]
            if m:
                fam_col = m[0]; break

        eps[ts_col] = pd.to_datetime(eps[ts_col], errors="coerce", utc=True)
        if eps[ts_col].isna().mean() > 0.5:
            eps[ts_col] = _robust_parse_epoch_series(eps[ts_col])
        eps = eps.dropna(subset=[ts_col])

        mapped_frames = []
        for repo_name, grp in eps.groupby(repo_col):
            base = Path(outdir) / repo_name.replace("/", "_")
            runs_fn = base / "runs.ndjson"
            if not runs_fn.exists():
                continue
            runs = pd.read_json(runs_fn, lines=True)
            for c in ["created_at", "run_started_at", "updated_at"]:
                if c in runs.columns:
                    runs[c] = pd.to_datetime(runs[c], errors="coerce", utc=True)

            for _, e in grp.iterrows():
                ts = e[ts_col]
                lo = ts - pd.Timedelta(days=window_days)
                hi = ts + pd.Timedelta(days=window_days)
                time_col = "run_started_at" if "run_started_at" in runs.columns else "created_at"
                sel = runs[(runs[time_col] >= lo) & (runs[time_col] <= hi)].copy()
                sel["episode_ts"] = ts
                if fam_col and fam_col in e:
                    sel["episode_family"] = e[fam_col]
                mapped_frames.append(sel)

        return pd.concat(mapped_frames, ignore_index=True) if mapped_frames else pd.DataFrame()

    mapped = map_episodes_to_runs(EPISODES_CSV, OUTDIR, window_days=WINDOW_DAYS)
    if not mapped.empty:
        out_parquet = Path(OUTDIR) / "episodes_to_runs.parquet"
        mapped.to_parquet(out_parquet, index=False)
        print(f"Mapped {len(mapped):,} rows → {out_parquet}")
    else:
        print("No episode→run mappings created (check that runs.ndjson exist for the repos).")
else:
    print("Mapping skipped (set MAP_EPISODES=True to enable and ensure episodes CSV exists).")


Config OK
Utils ready
GitHub client ready
Scrape logic loaded
Tokens: 6 found; Progress key: SCRAPE_PROGRESS_INDEX
Repos: 282; Since: 2024-01-01; Output: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ3\ci_out
Starting from SCRAPE_PROGRESS_INDEX=11 (12/282)
[12/282] BlueWallet/BlueWallet (token 6/6) — start
[12/282] BlueWallet/BlueWallet — done → SCRAPE_PROGRESS_INDEX=12
[13/282] CarlosMChica/easyrecycleradapters (token 1/6) — start
[13/282] CarlosMChica/easyrecycleradapters — done → SCRAPE_PROGRESS_INDEX=13
[14/282] CatimaLoyalty/Android (token 2/6) — start
[14/282] CatimaLoyalty/Android — done → SCRAPE_PROGRESS_INDEX=14
[15/282] Crazy-Marvin/MetadataRemover (token 3/6) — start
[15/282] Crazy-Marvin/MetadataRemover — done → SCRAPE_PROGRESS_INDEX=15
[16/282] DesarrolloAntonio/Shiori-Android-Client (token 4/6) — start
[16/282] DesarrolloAntonio/Shiori-Android-Client — done → SCRAPE_PROGRESS_INDEX=16
[17/282] DroidKaigi/conference-app-2020 (token 5/6) — start
[17/282] DroidKaigi/